In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ControlStateNet(nn.Module):
    def __init__(self, timesteps=128, feat_c=256, num_classes=3):
        super().__init__()
        self.timesteps = timesteps
        self.feat_c = feat_c
        self.num_classes = num_classes

        # ---------
        # Input 1 encoder: (B,3,224,224) -> (B,256,7,7)
        # ---------
        self.frame_encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 112

            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 56

            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 28

            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 14

            nn.Conv2d(256, feat_c, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 7
        )

        # In case you later tweak encoder, this forces channel count
        self.frame_proj = nn.Conv2d(feat_c, feat_c, kernel_size=1)

        # ---------
        # 3D processing: (B,256,T+1,7,7) -> (B,256,T+1,7,7)
        # ---------
        self.conv3d = nn.Sequential(
            nn.Conv3d(feat_c, feat_c, kernel_size=3, padding=1),
            nn.BatchNorm3d(feat_c),
            nn.ReLU(inplace=True),

            nn.Conv3d(feat_c, feat_c, kernel_size=3, padding=1),
            nn.BatchNorm3d(feat_c),
            nn.ReLU(inplace=True),

            # optional: more explicit temporal mixing
            nn.Conv3d(feat_c, feat_c, kernel_size=(3, 1, 1), padding=(1, 0, 0)),
            nn.ReLU(inplace=True),
        )

        # ---------
        # Classification head
        # ---------
        self.cls_head = nn.Sequential(
            nn.AdaptiveAvgPool3d((1, 1, 1)),  # -> (B,256,1,1,1)
            nn.Flatten(),                     # -> (B,256)
            nn.Linear(feat_c, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, num_classes),      # logits
        )

        # ---------
        # Next-state projection: keep (B,256,T,7,7)
        # ---------
        self.next_state_proj = nn.Conv3d(feat_c, feat_c, kernel_size=1)

    @staticmethod
    def init_state(batch_size, timesteps=128, feat_c=256, h=7, w=7, device=None, dtype=torch.float32):
        return torch.zeros((batch_size, feat_c, timesteps, h, w), device=device, dtype=dtype)

    def forward(self, frame_rgb, state):
        """
        frame_rgb: (B,3,224,224)
        state:     (B,256,T,7,7)

        returns:
          logits:     (B,3)
          next_state: (B,256,T,7,7)
        """

        B = frame_rgb.shape[0]

        # Encode frame -> (B,256,7,7)
        f = self.frame_encoder(frame_rgb)
        f = self.frame_proj(f)

        # Expand to depth=1 -> (B,256,1,7,7)
        f3 = f.unsqueeze(2)

        # Concat along depth -> (B,256,T+1,7,7)
        vol = torch.cat([state, f3], dim=2)

        # 3D processing
        v = self.conv3d(vol)

        # Classification logits
        logits = self.cls_head(v)

        # Next state: drop oldest slice, keep last T slices
        # v is (B,256,T+1,7,7) -> (B,256,T,7,7)
        next_state = v[:, :, 1:, :, :]

        # Project to keep channel count clean
        next_state = self.next_state_proj(next_state)

        return logits, next_state


In [2]:
import torch
import torch.nn as nn


class ControlStateNet(nn.Module):
    def __init__(self, timesteps=128, feat_c=256, num_classes=3):
        super().__init__()
        self.timesteps = timesteps
        self.feat_c = feat_c
        self.num_classes = num_classes

        # -------------------------
        # Input 1 encoder: (B,3,224,224) -> (B,256,7,7)
        # -------------------------
        self.frame_encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 112

            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 56

            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 28

            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 14

            nn.Conv2d(256, feat_c, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 7
        )
        self.frame_proj = nn.Conv2d(feat_c, feat_c, kernel_size=1)

        # -------------------------
        # State branch (keeps depth): (B,C,T+1,7,7) -> (B,C,T+1,7,7)
        # -------------------------
        self.conv3d_state = nn.Sequential(
            nn.Conv3d(feat_c, feat_c, kernel_size=3, padding=1),
            nn.BatchNorm3d(feat_c),
            nn.ReLU(inplace=True),

            nn.Conv3d(feat_c, feat_c, kernel_size=3, padding=1),
            nn.BatchNorm3d(feat_c),
            nn.ReLU(inplace=True),

            nn.Conv3d(feat_c, feat_c, kernel_size=(3, 1, 1), padding=(1, 0, 0)),
            nn.ReLU(inplace=True),
        )
        self.next_state_proj = nn.Conv3d(feat_c, feat_c, kernel_size=1)

        # -------------------------
        # Classification branch (temporal downsampling)
        # (B,C,T+1,7,7) -> compress along time -> pooled -> logits
        # -------------------------
        self.conv3d_cls = nn.Sequential(
            # local mixing
            nn.Conv3d(feat_c, feat_c, kernel_size=3, stride=(1, 1, 1), padding=1),
            nn.BatchNorm3d(feat_c),
            nn.ReLU(inplace=True),

            # downsample time: (T+1) -> ~((T+1)/2)
            nn.Conv3d(feat_c, feat_c, kernel_size=3, stride=(2, 1, 1), padding=1),
            nn.BatchNorm3d(feat_c),
            nn.ReLU(inplace=True),

            # more mixing
            nn.Conv3d(feat_c, feat_c, kernel_size=3, stride=(1, 1, 1), padding=1),
            nn.BatchNorm3d(feat_c),
            nn.ReLU(inplace=True),

            # downsample time again: -> ~((T+1)/4)
            nn.Conv3d(feat_c, feat_c, kernel_size=3, stride=(2, 1, 1), padding=1),
            nn.BatchNorm3d(feat_c),
            nn.ReLU(inplace=True),

            # more mixing
            nn.Conv3d(feat_c, feat_c, kernel_size=3, stride=(1, 1, 1), padding=1),
            nn.BatchNorm3d(feat_c),
            nn.ReLU(inplace=True),

            # downsample time again: -> ~((T+1)/8)
            nn.Conv3d(feat_c, feat_c, kernel_size=3, stride=(2, 1, 1), padding=1),
            nn.BatchNorm3d(feat_c),
            nn.ReLU(inplace=True),

            # more mixing
            nn.Conv3d(feat_c, feat_c, kernel_size=3, stride=(1, 1, 1), padding=1),
            nn.BatchNorm3d(feat_c),
            nn.ReLU(inplace=True),

            # downsample time again: -> ~((T+1)/16)
            nn.Conv3d(feat_c, feat_c, kernel_size=3, stride=(2, 1, 1), padding=1),
            nn.BatchNorm3d(feat_c),
            nn.ReLU(inplace=True),
        )

        self.cls_head = nn.Sequential(
            nn.AdaptiveAvgPool3d((1, 1, 1)),  # pool over (D,H,W)
            nn.Flatten(),                     # (B,C)
            nn.Linear(feat_c, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, num_classes),      # logits
        )

    @staticmethod
    def init_state(batch_size, timesteps=128, feat_c=256, h=7, w=7, device=None, dtype=torch.float32):
        return torch.zeros((batch_size, feat_c, timesteps, h, w), device=device, dtype=dtype)

    def forward(self, frame_rgb, state):
        """
        frame_rgb: (B,3,224,224)
        state:     (B,C,T,7,7)

        returns:
          logits:     (B,num_classes)
          next_state: (B,C,T,7,7)
        """
        # Encode frame -> (B,C,7,7)
        f = self.frame_proj(self.frame_encoder(frame_rgb))

        # Expand to depth=1 -> (B,C,1,7,7)
        f3 = f.unsqueeze(2)

        # Concat along depth -> (B,C,T+1,7,7)
        vol = torch.cat([state, f3], dim=2)

        # ---- State update branch (keeps full depth) ----
        v_state = self.conv3d_state(vol)              # (B,C,T+1,7,7)
        next_state = v_state[:, :, 1:, :, :]          # drop oldest -> (B,C,T,7,7)
        next_state = self.next_state_proj(next_state) # keep channels clean

        # ---- Classification branch (temporal downsample) ----
        v_cls = self.conv3d_cls(vol)
        logits = self.cls_head(v_cls)

        return logits, next_state


In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
ts = 512
ft = 256

model = ControlStateNet(timesteps=ts, feat_c=ft, num_classes=3).to(device)

B = 1
frame = torch.zeros((B, 3, 224, 224), device=device)
state = ControlStateNet.init_state(B, timesteps=ts, feat_c=ft, h=7, w=7, device=device)

logits, state = model(frame, state)

print("logits:", logits.shape)      # (1,3)
print("state:", state.shape)        # (1,256,128,7,7)


logits: torch.Size([1, 3])
state: torch.Size([1, 256, 512, 7, 7])


In [15]:
#print number of parameters
total_params = sum(p.numel() for p in model.parameters())   
print(f"Total parameters: {total_params}")

Total parameters: 1538115


In [5]:
import torch
import torch.nn as nn


class ControlCNNLSTM(nn.Module):
    def __init__(self, num_classes=3, feat_c=256, lstm_hidden=256, lstm_layers=1):
        super().__init__()

        # CNN encoder: (B,3,224,224) -> (B,feat_c,7,7)
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(256, feat_c, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

        # Pool to vector: (B,feat_c,7,7) -> (B,feat_c)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        # LSTM over time. We will feed one frame at a time as seq_len=1.
        self.lstm = nn.LSTM(
            input_size=feat_c,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,   # input: (B, seq, feat)
        )

        self.classifier = nn.Sequential(
            nn.Linear(lstm_hidden, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_classes),
        )

    def init_state(self, batch_size, device=None, dtype=torch.float32):
        device = device if device is not None else next(self.parameters()).device
        h0 = torch.zeros(self.lstm.num_layers, batch_size, self.lstm.hidden_size, device=device, dtype=dtype)
        c0 = torch.zeros(self.lstm.num_layers, batch_size, self.lstm.hidden_size, device=device, dtype=dtype)
        return (h0, c0)

    def forward(self, frame_rgb, state):
        """
        frame_rgb: (B,3,224,224)
        state: (h, c) where each is (num_layers, B, hidden)

        returns:
          logits: (B,num_classes)
          next_state: (h, c)
        """
        f = self.encoder(frame_rgb)                 # (B,feat_c,7,7)
        f = self.pool(f).flatten(1)                 # (B,feat_c)

        # Feed as a 1-step sequence
        f_seq = f.unsqueeze(1)                      # (B,1,feat_c)
        out, next_state = self.lstm(f_seq, state)   # out: (B,1,hidden)

        last = out[:, -1, :]                        # (B,hidden)
        logits = self.classifier(last)              # (B,num_classes)

        return logits, next_state


In [6]:
import torch

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Model hyperparameters
num_classes = 3          # Left, None, Right
feat_c = 256             # CNN feature channels
lstm_hidden = 256        # LSTM hidden size
lstm_layers = 1          # number of LSTM layers

# Create model
model = ControlCNNLSTM(
    num_classes=num_classes,
    feat_c=feat_c,
    lstm_hidden=lstm_hidden,
    lstm_layers=lstm_layers,
).to(device)

# Initialize LSTM state for batch_size = 1 (sequential frames)
state = model.init_state(batch_size=1, device=device)

print(model)
print("Initial h shape:", state[0].shape)
print("Initial c shape:", state[1].shape)


ControlCNNLSTM(
  (encoder): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (10): ReLU(inplace=True)
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (pool): A

In [7]:
# Dummy frame
frame = torch.zeros((1, 3, 224, 224), device=device)

logits, state = model(frame, state)

print("logits shape:", logits.shape)   # (1, 3)
print("next h shape:", state[0].shape)
print("next c shape:", state[1].shape)


logits shape: torch.Size([1, 3])
next h shape: torch.Size([1, 1, 256])
next c shape: torch.Size([1, 1, 256])


In [1]:
import csv
from pathlib import Path
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


class FrameControlSequenceDataset(Dataset):
    """
    Reads a CSV with columns: filename,left,right
    Loads frames sequentially from frames_dir.

    Target mapping (your rule):
      (0,0) -> None
      (1,1) -> None
      (1,0) -> Left
      (0,1) -> Right

    Returns:
      x: (3,H,W) float32 in [0,1]
      y: int64 class index {0:Left, 1:None, 2:Right}
      fname: filename
    """

    def __init__(self, csv_path: str, frames_dir: str, image_size=(224, 224)):
        self.csv_path = Path(csv_path)
        self.frames_dir = Path(frames_dir)

        if not self.csv_path.exists():
            raise FileNotFoundError(f"CSV not found: {self.csv_path}")
        if not self.frames_dir.exists():
            raise FileNotFoundError(f"Frames dir not found: {self.frames_dir}")

        self.transform = transforms.Compose([
            transforms.Resize(image_size),
            transforms.ToTensor(),  # /255 -> [0,1]
        ])

        # Read CSV rows
        rows = []
        with open(self.csv_path, "r", newline="") as f:
            reader = csv.DictReader(f)
            for r in reader:
                fname = r["filename"]
                left = int(r["left"])
                right = int(r["right"])
                rows.append((fname, left, right))

        # Sort explicitly by filename to preserve frame order
        rows.sort(key=lambda t: t[0])
        self.rows = rows

        # Class mapping
        self.class_names = ["Left", "None", "Right"]

    def __len__(self):
        return len(self.rows)

    def _to_class_index(self, left: int, right: int) -> int:
        # 0: Left, 1: None, 2: Right
        if (left == 0 and right == 0) or (left == 1 and right == 1):
            return 1  # None
        if left == 1 and right == 0:
            return 0  # Left
        if left == 0 and right == 1:
            return 2  # Right
        # Fallback (shouldn't happen)
        return 1

    def __getitem__(self, idx):
        fname, left, right = self.rows[idx]
        img_path = self.frames_dir / fname
        if not img_path.exists():
            raise FileNotFoundError(f"Missing frame: {img_path}")

        with Image.open(img_path) as im:
            im = im.convert("RGB")
            x = self.transform(im)

        y = torch.tensor(self._to_class_index(left, right), dtype=torch.long)
        return x, y, fname
    

import csv
import random
from pathlib import Path
from PIL import Image

import torch
from torch.utils.data import Dataset
from torchvision import transforms


class FrameControlSequenceDataset(Dataset):
    """
    Sequence dataset with controlled class imbalance.

    - Keeps ALL Left and Right frames
    - Samples up to `max_none_per_epoch` None frames per epoch
    - Resampling happens by calling `set_epoch(epoch)`
    - Order is preserved (important for temporal models)
    """

    def __init__(
        self,
        csv_path: str,
        frames_dir: str,
        image_size=(224, 224),
        max_none_per_epoch: int = 30_000,
        seed: int = 42,
    ):
        self.csv_path = Path(csv_path)
        self.frames_dir = Path(frames_dir)
        self.max_none_per_epoch = max_none_per_epoch
        self.seed = seed

        self.transform = transforms.Compose([
            transforms.Resize(image_size),
            transforms.ToTensor(),  # /255
        ])

        # Load CSV
        rows = []
        with open(self.csv_path, "r", newline="") as f:
            reader = csv.DictReader(f)
            for r in reader:
                fname = r["filename"]
                left = int(r["left"])
                right = int(r["right"])
                rows.append((fname, left, right))

        # Sort by filename to preserve temporal order
        rows.sort(key=lambda t: t[0])
        self.rows = rows

        # Split indices by class
        self.left_idx = []
        self.right_idx = []
        self.none_idx = []

        for i, (_fname, left, right) in enumerate(self.rows):
            if left == 1 and right == 0:
                self.left_idx.append(i)
            elif left == 0 and right == 1:
                self.right_idx.append(i)
            else:
                self.none_idx.append(i)

        self.class_names = ["Left", "None", "Right"]

        # Build first epoch index list
        self.set_epoch(0)

    def set_epoch(self, epoch: int):
        """
        Call this at the start of every epoch to resample None frames.
        """
        rng = random.Random(self.seed + epoch)

        # Sample None indices
        if len(self.none_idx) > self.max_none_per_epoch:
            sampled_none = rng.sample(self.none_idx, self.max_none_per_epoch)
        else:
            sampled_none = list(self.none_idx)

        # Combine all indices and keep temporal order
        active = set(self.left_idx + self.right_idx + sampled_none)
        self.active_indices = [i for i in range(len(self.rows)) if i in active]

    def __len__(self):
        return len(self.active_indices)

    def _to_class_index(self, left: int, right: int) -> int:
        # 0: Left, 1: None, 2: Right
        if (left == 0 and right == 0) or (left == 1 and right == 1):
            return 1
        if left == 1 and right == 0:
            return 0
        if left == 0 and right == 1:
            return 2
        return 1

    def __getitem__(self, idx):
        real_idx = self.active_indices[idx]
        fname, left, right = self.rows[real_idx]

        img_path = self.frames_dir / fname
        if not img_path.exists():
            raise FileNotFoundError(f"Missing frame: {img_path}")

        with Image.open(img_path) as im:
            im = im.convert("RGB")
            x = self.transform(im)

        y = torch.tensor(self._to_class_index(left, right), dtype=torch.long)
        return x, y, fname



In [3]:
csv_path = r"D:\Data\Game Dataset\az_recorder_20260103_125453s_224x101_controls_extracted.csv"
frames_dir = r"D:\Data\Game Dataset\az_recorder_20260103_125453s_224x101"

ds = FrameControlSequenceDataset(csv_path=csv_path, frames_dir=frames_dir, image_size=(224, 224))
dl = DataLoader(ds, batch_size=1, shuffle=False)


In [11]:
import time
import torch
import torch.nn as nn


def train_cnn_lstm_classification_only(
    model,
    dataloader,
    class_names,
    epochs=5,
    lr=1e-4,
    weight_decay=1e-4,
    device=None,
    print_every=50,
    save_path="control_cnn_lstm.pt",
):
    """
    Stateful training loop for CNN+LSTM model.
    Trains only on classification logits.

    Assumes dataloader yields: (x, y, fname)
      x: (B,3,224,224)  usually B=1
      y: (B,) int64 class index: 0 Left, 1 None, 2 Right

    LSTM state:
      state = (h, c) where h,c are (num_layers, B, hidden)
    """

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(1, epochs + 1):
        ds.set_epoch(epoch)
        model.train()

        # Initialize LSTM state at start of epoch/sequence
        state = model.init_state(batch_size=1, device=device)

        running_loss = 0.0
        running_correct = 0
        seen = 0

        t0 = time.time()

        for step, batch in enumerate(dataloader, start=1):
            x, y, _fname = batch

            x = x.to(device)
            y = y.to(device).view(-1)

            B = x.shape[0]

            # If batch_size changes, re-init state to match
            h, c = state
            if h.shape[1] != B:
                state = model.init_state(batch_size=B, device=device)

            # Detach state to prevent backprop through time
            h, c = state
            state = (h.detach(), c.detach())

            optimizer.zero_grad(set_to_none=True)

            logits, next_state = model(x, state)

            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            # Carry forward state
            state = next_state

            # Stats
            preds = logits.argmax(dim=1)
            running_loss += loss.item() * B
            running_correct += (preds == y).sum().item()
            seen += B

            if (step % print_every == 0) or (step == 1) or (step == len(dataloader)):
                avg_loss = running_loss / max(seen, 1)
                avg_acc = running_correct / max(seen, 1)

                tgt_name = class_names[y.item()]
                pred_name = class_names[preds.item()]

                print(
                    f"Epoch {epoch:02d}/{epochs} | "
                    f"Step {step:05d}/{len(dataloader):05d} | "
                    f"loss={avg_loss:.4f} acc={avg_acc:.4f} | "
                    f"target={tgt_name} pred={pred_name}"
                )

        epoch_time = time.time() - t0
        epoch_loss = running_loss / max(seen, 1)
        epoch_acc = running_correct / max(seen, 1)

        print(
            f"Epoch {epoch:02d} done | "
            f"loss={epoch_loss:.4f} acc={epoch_acc:.4f} | "
            f"time={epoch_time:.1f}s"
        )

    # Save checkpoint
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "class_names": class_names,
        },
        save_path,
    )
    print(f"Saved model to: {save_path}")

    return model


In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"

trained_model = train_cnn_lstm_classification_only(
    model=model,
    dataloader=dl,  # DataLoader(ds, batch_size=1, shuffle=False)
    class_names=["Left", "None", "Right"],
    epochs=10,
    lr=1e-4,
    print_every=10,
    device=device,
    save_path="control_cnn_lstm.pt",
)


Epoch 01/10 | Step 00001/65476 | loss=1.0896 acc=1.0000 | target=None pred=None
Epoch 01/10 | Step 00010/65476 | loss=1.0696 acc=1.0000 | target=None pred=None
Epoch 01/10 | Step 00020/65476 | loss=1.0243 acc=1.0000 | target=None pred=None
Epoch 01/10 | Step 00030/65476 | loss=0.8927 acc=1.0000 | target=None pred=None
Epoch 01/10 | Step 00040/65476 | loss=0.7021 acc=1.0000 | target=None pred=None
Epoch 01/10 | Step 00050/65476 | loss=0.5662 acc=1.0000 | target=None pred=None
Epoch 01/10 | Step 00060/65476 | loss=0.4731 acc=1.0000 | target=None pred=None
Epoch 01/10 | Step 00070/65476 | loss=0.4061 acc=1.0000 | target=None pred=None
Epoch 01/10 | Step 00080/65476 | loss=0.3558 acc=1.0000 | target=None pred=None
Epoch 01/10 | Step 00090/65476 | loss=0.3166 acc=1.0000 | target=None pred=None
Epoch 01/10 | Step 00100/65476 | loss=0.2851 acc=1.0000 | target=None pred=None
Epoch 01/10 | Step 00110/65476 | loss=0.2594 acc=1.0000 | target=None pred=None
Epoch 01/10 | Step 00120/65476 | loss=0.

In [8]:
import time
import torch
import torch.nn as nn


def train_state_model_classification_only(
    model,
    dataloader,
    epochs=5,
    lr=1e-4,
    weight_decay=1e-4,
    device=None,
    timesteps=128,
    feat_c=256,
    h=7,
    w=7,
    print_every=50,
    save_path="control_state_net.pt",
):
    """
    Trains model on classification logits only.
    Internal state is still fed forward, but:
      - we do NOT apply any loss to next_state
      - we DETACH state at every step (stop-grad through time)

    Assumes dataloader yields: (x, y, fname)
      x: (B,3,224,224) typically B=1
      y: (B,) class index: 0 Left, 1 None, 2 Right
    """

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(1, epochs + 1):
        model.train()

        # Reset state at the start of each epoch/sequence
        state = torch.zeros((1, feat_c, timesteps, h, w), device=device, dtype=torch.float32)

        running_loss = 0.0
        running_correct = 0
        seen = 0

        t0 = time.time()

        for step, batch in enumerate(dataloader, start=1):
            x, y, _fname = batch

            # Usually batch_size=1 for sequential processing
            x = x.to(device)
            y = y.to(device).view(-1)

            # If someone sets batch_size>1, this keeps things consistent:
            B = x.shape[0]
            if state.shape[0] != B:
                state = torch.zeros((B, feat_c, timesteps, h, w), device=device, dtype=torch.float32)

            # Stop gradient through time (important)
            state = state.detach()

            optimizer.zero_grad(set_to_none=True)

            logits, next_state = model(x, state)

            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            # Move state forward (no loss on it)
            state = next_state

            # Stats
            running_loss += loss.item() * B
            preds = logits.argmax(dim=1)
            running_correct += (preds == y).sum().item()
            seen += B

            if (step % print_every == 0) or (step == 1) or (step == len(dataloader)):
                avg_loss = running_loss / max(seen, 1)
                avg_acc = running_correct / max(seen, 1)
                print(
                    f"Epoch {epoch:02d}/{epochs} | "
                    f"Step {step:05d}/{len(dataloader):05d} | "
                    f"loss={avg_loss:.4f} acc={avg_acc:.4f}"
                )

        epoch_time = time.time() - t0
        epoch_loss = running_loss / max(seen, 1)
        epoch_acc = running_correct / max(seen, 1)

        print(
            f"Epoch {epoch:02d} done | "
            f"loss={epoch_loss:.4f} acc={epoch_acc:.4f} | "
            f"time={epoch_time:.1f}s"
        )

    # Save checkpoint
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "timesteps": timesteps,
            "feat_c": feat_c,
            "h": h,
            "w": w,
        },
        save_path,
    )
    print(f"Saved model to: {save_path}")

    return model


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

trained_model = train_state_model_classification_only(
    model=model,
    dataloader=dl,          # your DataLoader(shuffle=False, batch_size=1)
    epochs=10,
    lr=1e-4,
    device=device,
    timesteps=128,
    feat_c=256,
    h=7,
    w=7,
    print_every=10,
    save_path="control_state_net.pt",
)


In [ ]:
import time
import torch
import torch.nn as nn


def train_state_model_classification_only(
    model,
    dataloader,
    class_names,              # <-- added
    epochs=5,
    lr=1e-4,
    weight_decay=1e-4,
    device=None,
    timesteps=128,
    feat_c=256,
    h=7,
    w=7,
    print_every=50,
    save_path="control_state_net.pt",
):
    """
    Trains model on classification logits only.
    Prints target and predicted class at progress steps.
    """

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    weights = torch.tensor([1.0, 0.4, 1.0], device=device)
    criterion = nn.CrossEntropyLoss(weight=weights)

    #criterion = nn.CrossEntropyLoss()

    for epoch in range(1, epochs + 1):
        ds.set_epoch(epoch)
        model.train()

        # Initial internal state
        state = torch.zeros((1, feat_c, timesteps, h, w), device=device, dtype=torch.float32)

        running_loss = 0.0
        running_correct = 0
        seen = 0

        t0 = time.time()

        for step, batch in enumerate(dataloader, start=1):
            x, y, _fname = batch

            x = x.to(device)
            y = y.to(device).view(-1)

            B = x.shape[0]
            if state.shape[0] != B:
                state = torch.zeros((B, feat_c, timesteps, h, w), device=device, dtype=torch.float32)

            # Detach state to stop backprop through time
            state = state.detach()

            optimizer.zero_grad(set_to_none=True)

            logits, next_state = model(x, state)

            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            # Move state forward
            state = next_state

            # Stats
            preds = logits.argmax(dim=1)
            running_loss += loss.item() * B
            running_correct += (preds == y).sum().item()
            seen += B

            # 🔹 Progress print with target & prediction
            if (step % print_every == 0) or (step == 1) or (step == len(dataloader)):
                avg_loss = running_loss / max(seen, 1)
                avg_acc = running_correct / max(seen, 1)

                tgt_name = class_names[y.item()]
                pred_name = class_names[preds.item()]

                print(
                    f"Epoch {epoch:02d}/{epochs} | "
                    f"Step {step:05d}/{len(dataloader):05d} | "
                    f"loss={avg_loss:.4f} acc={avg_acc:.4f} | "
                    f"target={tgt_name} pred={pred_name}"
                )

        epoch_time = time.time() - t0
        epoch_loss = running_loss / max(seen, 1)
        epoch_acc = running_correct / max(seen, 1)

        print(
            f"Epoch {epoch:02d} done | "
            f"loss={epoch_loss:.4f} acc={epoch_acc:.4f} | "
            f"time={epoch_time:.1f}s"
        )

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "timesteps": timesteps,
            "feat_c": feat_c,
            "h": h,
            "w": w,
        },
        save_path,
    )
    print(f"Saved model to: {save_path}")

    return model


In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"

trained_model = train_state_model_classification_only(
    model=model,
    dataloader=dl,          # your DataLoader(shuffle=False, batch_size=1)
    class_names=["Left", "None", "Right"],
    epochs=10,
    lr=1e-4,
    device=device,
    timesteps=ts,
    feat_c=ft,
    h=7,
    w=7,
    print_every=10,
    save_path="control_state_net.pt",
)

NameError: name 'ts' is not defined

In [9]:
trained_model = train_state_model_classification_only(
    model=model,
    dataloader=dl,
    class_names=["Left", "None", "Right"],
    epochs=10,
    lr=1e-4,
    print_every=10,
)


IndexError: index 1 is out of bounds for dimension 0 with size 1

In [10]:
trained_model = train_state_model_classification_only(
    model=model,
    dataloader=dl,
    class_names=["Left", "None", "Right"],
    epochs=10,
    lr=1e-4,
    print_every=10,
)


IndexError: index 1 is out of bounds for dimension 0 with size 1